In [32]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [ ]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
from flatten_json import flatten

In [34]:
table_str1 = ("`hitwicketsuperstars.analytics_190927423.new_user_reference`")
print(table_str1)

`hitwicketsuperstars.analytics_190927423.new_user_reference`


In [35]:
client = bigquery.Client.from_service_account_json('/home/analytics/.secure_files/hitwicketsuperstars-f3e8c620a88c.json')
query =( f"""SELECT
*
FROM
      {table_str1}"""
       )
df = client.query(query).to_dataframe()
df.head()

,device_id,user_first_touch_timestamp,platform
0,4D4C71A9-2DF1-4312-817B-B61DEE412C97,2019-05-21 08:17:04+00:00,IOS
1,1BF17F02-FCDF-427A-B818-8D8D073E440D,2019-05-13 16:52:52+00:00,IOS
2,2CA5977E-5A93-45A2-A637-0E887149D227,2019-05-06 07:06:12+00:00,IOS
3,7359B1DA-F24B-4DD2-8C8E-EC3B26B9A66D,2019-05-13 06:04:20+00:00,IOS
4,6EA89DC3-86E3-43D0-ADEA-EE830F4F7B33,2019-04-24 12:58:00+00:00,IOS


In [36]:
original_df = df.copy()

In [37]:
df = original_df.copy()

In [38]:
len(df)

552989

In [39]:
df = df.sort_values(['device_id','user_first_touch_timestamp'])
df = df.drop_duplicates('device_id')

In [40]:
len(df)

552988

In [41]:
df.head()

,device_id,user_first_touch_timestamp,platform
35623,,2019-04-17 02:29:26+00:00,ANDROID
42766,000028f181ded4525f5fca6fb586287e,2019-04-07 08:47:19+00:00,ANDROID
15630,00006eff08202bed28328993d1478d35,2019-04-09 00:51:27+00:00,ANDROID
96662,00009f428c7e600c5a9bf02876a24dac,2019-04-23 09:15:13+00:00,ANDROID
7402,0000db0501b7a9e930ea14d091257b42,2019-04-07 00:44:39+00:00,ANDROID


In [42]:
df.columns = ['device_id','first_touch','platform']

In [43]:
df['first_touch'] = df['first_touch'].dt.tz_localize(None)

In [44]:
df.head()

,device_id,first_touch,platform
35623,,2019-04-17 02:29:26,ANDROID
42766,000028f181ded4525f5fca6fb586287e,2019-04-07 08:47:19,ANDROID
15630,00006eff08202bed28328993d1478d35,2019-04-09 00:51:27,ANDROID
96662,00009f428c7e600c5a9bf02876a24dac,2019-04-23 09:15:13,ANDROID
7402,0000db0501b7a9e930ea14d091257b42,2019-04-07 00:44:39,ANDROID


In [45]:
len(df)

552988

In [56]:
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname'])
c_users = cursor.superstars.users
aw_users = []
for documents in c_users.find({},{"sign_up_details":1, "created_at":1}):
    aw_users.append(documents)
dic_flattened = [flatten(d) for d in aw_users]
df_users = pd.DataFrame(dic_flattened)
users = df_users[["_id","created_at","sign_up_details_device_id"]]
users.columns = ["user_id","create_time","device_id"]

In [57]:
users = users.sort_values(['device_id','create_time'])
users = users.drop_duplicates('device_id')

In [58]:
users.head()

,user_id,create_time,device_id
42294,5cac52eb3be71932baa04f41,2019-04-09 08:08:11.452,00006eff08202bed28328993d1478d35
58833,5cae02741d71502e0f2b965c,2019-04-10 14:49:24.128,0000e8d7d09604feae288ba6c73c0843
58481,5cadfcd67c3f272e09d999ae,2019-04-10 14:25:26.180,000197332b0ffd68d947a16e14a16318
48195,5cace6057925c0061cd18287,2019-04-09 18:35:49.110,00020b99801ce0c99dba75af43a9ef12
30885,5caac638b032434e461b39da,2019-04-08 03:55:36.350,0002db49ca444e49fa34b6891ebcc13a


In [83]:
query = (
    f"""SELECT
  user_id,
  device.mobile_os_hardware_model as device
FROM `hitwicketsuperstars.analytics_190927423.events_2019*`
      GROUP BY user_id, device
      """
)
df_all = client.query(query).to_dataframe()

In [84]:
df_all.head()

,user_id,device
0,3d39f3e8dfea4aee78ac0eb1dc0bef2a,SM-E700H
1,2a6936d898088741b0733f357e9f926f,X1S
2,0eae962310390364a3c3c90c3bc15f8f,RMX1825
3,281d594dae109383dff19ec42f1ac4be,SM-G935F
4,ff5ba3679bd4779d82cce4b73e71cc09,SM-G920F


In [85]:
len(df_all)

556353

In [86]:
users1 = pd.merge(users[['device_id','create_time']],df_all,left_on='device_id',right_on='user_id', how='left')

In [87]:
users1.head()

,device_id,create_time,user_id,device
0,00006eff08202bed28328993d1478d35,2019-04-09 08:08:11.452,00006eff08202bed28328993d1478d35,Redmi 4A
1,0000e8d7d09604feae288ba6c73c0843,2019-04-10 14:49:24.128,0000e8d7d09604feae288ba6c73c0843,vivo 1723
2,000197332b0ffd68d947a16e14a16318,2019-04-10 14:25:26.180,000197332b0ffd68d947a16e14a16318,SM-J810G
3,00020b99801ce0c99dba75af43a9ef12,2019-04-09 18:35:49.110,00020b99801ce0c99dba75af43a9ef12,MotoG3
4,0002db49ca444e49fa34b6891ebcc13a,2019-04-08 03:55:36.350,0002db49ca444e49fa34b6891ebcc13a,SM-M205F


In [88]:
ftue_completed = pd.merge(df,users1,on='device_id')

In [89]:
len(ftue_completed)

128104

In [90]:
ftue_completed['time_taken'] = (ftue_completed['create_time'] - ftue_completed['first_touch']).dt.total_seconds()

In [91]:
print(len(ftue_completed))
ftue_completed.head()

128104


,device_id,first_touch,platform,create_time,user_id,device,time_taken
0,00006eff08202bed28328993d1478d35,2019-04-09 00:51:27,ANDROID,2019-04-09 08:08:11.452,00006eff08202bed28328993d1478d35,Redmi 4A,26204.452
1,0000e8d7d09604feae288ba6c73c0843,2019-04-10 12:15:59,ANDROID,2019-04-10 14:49:24.128,0000e8d7d09604feae288ba6c73c0843,vivo 1723,9205.128
2,000197332b0ffd68d947a16e14a16318,2019-04-10 14:20:23,ANDROID,2019-04-10 14:25:26.180,000197332b0ffd68d947a16e14a16318,SM-J810G,303.180
3,00020b99801ce0c99dba75af43a9ef12,2019-04-09 23:37:51,ANDROID,2019-04-09 18:35:49.110,00020b99801ce0c99dba75af43a9ef12,MotoG3,-18121.890
4,0002db49ca444e49fa34b6891ebcc13a,2019-04-08 03:50:44,ANDROID,2019-04-08 03:55:36.350,0002db49ca444e49fa34b6891ebcc13a,SM-M205F,292.350


In [92]:
ftue_completed = ftue_completed[ftue_completed['time_taken']>0]

In [93]:
len(ftue_completed)

124276

In [94]:
top_phones = ['Redmi Note 4','SM-J701F','Redmi 5A','Redmi 6A','SM-G610F','Redmi 4','Redmi Note 5 Pro','SM-J600G','SM-J250F','Redmi Y2']

In [95]:
ftue_completed = ftue_completed[ftue_completed["device"].isin(top_phones)]

In [96]:
len(ftue_completed)

31917

In [102]:
device_med = ftue_completed.groupby('device').agg({'time_taken':np.median}).reset_index()

In [103]:
device_med.set_index('device',inplace=True)
device_med.head(10)

,time_taken
device,
Redmi 4,430.4470
Redmi 5A,523.8630
Redmi 6A,472.8910
Redmi Note 4,404.8910
Redmi Note 5 Pro,353.7730
Redmi Y2,370.7330
SM-G610F,412.2985
SM-J250F,442.3985
SM-J600G,392.8625
